# Tokenizers
> Imagine a word is an interval of time series data...   

In [ ]:
#| default_exp tokenizers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn.functional as F, torch.nn as nn
from physiojepa.layers import Patch, InceptionBlock

In [ ]:
#| export
class TS_Tokenizer(nn.Module):
    """
    Tokenizer class based on a Conv1D
    ---
        c_in (int): Number of input channels
        patch_size (int): Size of each patch/kernel
        d_model (int): Output embedding dimension
    """
    def __init__(self, c_in, patch_size, d_model, patch_stride=None, shared_embedding=True):
        super().__init__()
        self.c_in = c_in
        self.d_model = d_model
        self.patch_size = patch_size
        self.patch_stride = patch_stride if patch_stride is not None else patch_size
        self.shared_embedding = shared_embedding
        if not shared_embedding:
            assert d_model % c_in == 0, f"d_model ({d_model}) must be divisible by c_in ({c_in})"
        self.proj = nn.Conv1d(
            in_channels=c_in,
            out_channels=d_model, # if shared embedding, then d_model is the output dimension
            kernel_size=self.patch_size,
            stride=self.patch_stride,
            padding=0,
            groups=c_in if not shared_embedding else 1 # added to handle multiple channels / keep them separate
        )

    def forward(self, x):
        """
        Args:
            x: Either a regular tensor [batch_size, C, L] 
        Returns:
            Regular tensor [batch_size, num_patches, d_model]
        """
        bs, C, seq_len = x.shape
        if ((seq_len-self.patch_size) % self.patch_size != 0):
            # only pad if remainder
            x = F.pad(x, (0, self.patch_size), 'constant', value=0.) # pad at the end with value
        x = self.proj(x)  # [batch_size, d_model * c_in, num_patches]
        if not self.shared_embedding:
            x = x.reshape(bs, self.c_in, self.d_model // self.c_in if not self.shared_embedding else self.d_model, -1)
            x = x.permute(0, 3, 1, 2) # [batch_size, num_patches, n_vars, d_model]
        else:
            x = x.transpose(1, 2) # [batch_size, num_patches, d_model]
        return x

class TS_Tokenizer_Complex(nn.Module):
    """
    Time series 2D convolutional Embedding
    """
    def __init__(
        self,
        c_in,
        patch_size,
        d_model,
        constant_pad_value=0.
    ):
        super().__init__()
        self.patch_size = patch_size
        self.constant_pad_value = constant_pad_value
        #self.simple_conv = nn.Conv1d(c_in, d_model, kernel_size=patch_size, stride=patch_size)
        self.emb_1 = nn.Conv2d(1, d_model*4, kernel_size=[1, patch_size], stride=[1, patch_size])
        self.emb_2 = nn.Conv2d(d_model*4, d_model, kernel_size=[c_in, 1])
    def forward(self, x):
        """
        Input: [bs x n channels x seq_len]
        Out:  [bs x num_patch x d_model]
        """
        bs, C, seq_len = x.shape
        #x = self.simple_conv(x).transpose(1, 2)
        if ((seq_len-self.patch_size) % self.patch_size != 0):
            # only pad if remainder
            x = F.pad(x, (0, self.patch_size), 'constant', value=self.constant_pad_value) # pad at the end with value
        x = x.unsqueeze(1)
        x = self.emb_1(x) # [bs x d_model*4 x n channels x n patches]
        x = self.emb_2(x) # [bs x d_model x 1 x n patches]
        x = x.transpose(1, 3) 
        return x.squeeze(2)
    
class LinearTokenizer(nn.Module):
    def __init__(self, 
                 c_in, # the number of input channels
                 patch_size, # the length of the patches (either stft or interval length)
                 d_model, # the dimension of the initial linear layers for inputting patches into transformer
                 shared_embedding=False, # indicator of whether to project each channel individually or together
                 ):
        super().__init__()

        self.shared_embedding = shared_embedding
        self.n_vars = c_in
        self.patch_size = patch_size
        self.d_model = d_model
        self.patch = Patch(patch_len=self.patch_size, stride=self.patch_size)
        # Input encoding: projection of feature vectors onto a d-dim vector space
        ## note that this could be an MLP too, if you want
        if not shared_embedding:
            self.W_P = nn.ModuleList()
            for _ in range(self.n_vars): self.W_P.append(nn.Linear(patch_size, d_model))
        else:
            self.W_P = nn.Linear(patch_size, d_model)
        #self.activation = nn.GELU()
        #self.project_out = nn.Linear(d_model*c_in, d_model)

    def forward(self, x):          
        """
        input: x: tensor [bs x nvars x seq_len]
        returns: x: tensor [bs x num_patch x d_model]
        """
        # Input embedding
        bs = x.shape[0]
        x = self.patch(x, constant_pad=True, constant_pad_value=0) # [bs x num_patch x n_vars x patch_len]
        if not self.shared_embedding:
            x_out = []
            for i in range(self.n_vars):
                x_out.append(self.W_P[i](x[:,:,i,:]))
            x = torch.stack(x_out, dim=2)
        else:
            x = self.W_P(x) # x: [bs x num_patch x nvars x d_model]

        #x = self.activation(x)
        x = x.reshape(bs, -1, self.d_model)
        #x = x.flatten(start_dim=-2) # bs x n patch x nvars * d_model
        #x = self.project_out(x)
        return x 
    
class InceptionTokenizer(nn.Module):
    def __init__(self, 
                 c_in, # the number of input channels
                 patch_size, # the length of the patches (either stft or interval length)
                 d_model, # the dimension of the initial linear layers for inputting patches into transformer
                 patch_stride=None, # the stride of the patches
                 shared_embedding=True,
                 **tokenizer_kwargs
                 ):
        super().__init__()
        self.n_vars = c_in
        self.patch_size = patch_size
        self.patch_stride = patch_stride if patch_stride is not None else patch_size
        self.d_model = d_model
        self.shared_embedding = shared_embedding
        self.inception = InceptionBlock(in_channels=c_in, groups=c_in if not shared_embedding else 1, **tokenizer_kwargs)
        self.inception_out = self.inception.bottleneck_channels * 4
        if not shared_embedding:
            assert d_model % c_in == 0, f"d_model ({d_model}) must be divisible by c_in ({c_in})"

        self.patch = nn.Conv1d(
            in_channels=self.inception_out,
            out_channels=d_model,
            kernel_size=self.patch_size,
            stride=self.patch_stride,
            padding=0,
            groups=self.n_vars if not shared_embedding else 1 # added to handle multiple channels / keep them separate
        )

    def forward(self, x):
        """
        input: x: tensor [bs x n channels x seq_len]
        returns: x: tensor [bs x num_patch x d_model]
        """
        bs, C, seq_len = x.shape
        if ((seq_len-self.patch_size) % self.patch_size != 0):
            # only pad if remainder
            x = F.pad(x, (0, self.patch_size), 'constant', value=0.) # pad at the end with value
        x = self.inception(x) # [bs x d_model * n_vars (if not shared embedding) else d_model x seq_len]
        x = self.patch(x)
        if not self.shared_embedding:
            x = x.reshape(bs, self.n_vars, self.d_model // self.n_vars if not self.shared_embedding else self.d_model, -1)
            x = x.permute(0, 3, 1, 2)
        else:
            x = x.transpose(1, 2) # [bs x num_patch x num_vars xd_model]
        return x


In [ ]:
#| export
class PatchEncoder(nn.Module):
    def __init__(self, 
                 c_in, # the number of input channels
                 patch_len, # the length of the patches (either stft or interval length)
                 d_model, # the dimension of the initial linear layers for inputting patches into transformer
                 shared_embedding, # indicator of whether to project each channel individually or together
                 ):
        super().__init__()

        self.shared_embedding = shared_embedding
        self.n_vars = c_in
        self.patch_len = patch_len
        self.d_model = d_model

        # Input encoding: projection of feature vectors onto a d-dim vector space
        ## note that this could be an MLP too, if you want
        if not shared_embedding:
            self.W_P = nn.ModuleList()
            for _ in range(self.n_vars): self.W_P.append(nn.Linear(patch_len, d_model))
        else:
            self.W_P = nn.Linear(patch_len, d_model)

    def forward(self, x):          
        """
        input: x: tensor [bs x num_patch x nvars x patch_len]
        returns: x: tensor [bs x num_patch x nvars x d_model]
        """
        # Input embedding
        if not self.shared_embedding:
            x_out = []
            for i in range(self.n_vars):
                x_out.append(self.W_P[i](x[:,:,i,:]))
            x = torch.stack(x_out, dim=2)
        else:
            x = self.W_P(x) # x: [bs x num_patch x nvars x d_model]
        return x

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()